<a href="https://colab.research.google.com/github/Eliezer-Carvalho/Adamastor-GPT/blob/main/Adamastor_GPT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### **Adamastor-GPT**

Este modelo é baseado na arquitetura <b> Transformer </b>, inspirado nos modelos do tipo <b> GPT (<i>Generative Pre-trained Transformer</i>)</b>. <br>

Ao longo deste Colab, iremos navegar pelas componentes cruciais dos modelos GPT-like, que atualmente dominam grande parte do mundo da Inteligência Artificial.

<b> No final, teremos um modelo estilo GPT funcional, capaz de realizar inferência de forma limpa e consistente. </b>

In [59]:
from tokenizers import Tokenizer
import torch
import torch.nn as nn
import torch.nn.functional as F

device = "cuda" if torch.cuda.is_available() else "cpu" #Mover tudo para a GPU #Fulcral para treino

### Dados do Modelo

In [60]:
Sequence_Length = 32
Batch_Size = 16
Embedding_Dimension = 64
Head_Size = 8 #Head_Size = Embedding_Dimension // Number_Head
Number_Head = 8

###


---

### Tokenization


In [69]:
Tokenizer = Tokenizer.from_file ("GAMA")

with open ("Os Lusiadas.txt", "r", encoding = "utf-8") as f:
  dataset = f.read()

Tokenization = Tokenizer.encode (dataset)
print (len(Tokenization.ids))
print (len(Tokenization.tokens))

tensor = torch.tensor (Tokenization.ids, dtype = torch.long)
print (tensor [:50])

n = int (0.9 * (len(tensor)))
dados_treino = tensor [:n] #90% Dados de Treino
dados_teste = tensor [n:] #10% Dados de Teste

75042
75042
tensor([  27,   31,    3,   24,   33,   31,   70,   14,   17,   14,   31,    3,
           0,   24,   58,   83,   95,  692,  120,  541,    0, 2342,   22,    3,
           0,  344,  672, 1062, 2563,  541, 2461,  398,  143,  138, 2854,  115,
         208, 4182,  783, 3295, 2515,  575, 4573,  694,  398, 1999,  662,   39,
         543,  114])


###


---

### Batch e Sequence Length

In [71]:
torch.manual_seed (2000)

def Batch ():
  idx = torch.randint (0, len(dados_treino) - Sequence_Length, (Batch_Size, ))
  x = torch.stack ([dados_treino [i: i + Sequence_Length] for i in idx])
  y = torch.stack ([dados_treino [i + 1: i + Sequence_Length + 1] for i in idx])

  return x, y

x, y = Batch()

x = x.to(device) #Passar para GPU
print (x.shape) #(B, T)
print (x[1])

y = y.to(device) #Passar para GPU
print (y.shape) #(B, T)
print (y[1])

torch.Size([16, 32])
tensor([ 106, 1249, 1065,  875,  554, 3649,  878, 1657, 1544, 1065, 1308,  290,
        2747,  207, 4168,  264,  170,  162,  277,  166,  396,  909, 2662, 3882,
        3426,  233, 1395, 3344, 3492,   94, 2314,  534])
torch.Size([16, 32])
tensor([1249, 1065,  875,  554, 3649,  878, 1657, 1544, 1065, 1308,  290, 2747,
         207, 4168,  264,  170,  162,  277,  166,  396,  909, 2662, 3882, 3426,
         233, 1395, 3344, 3492,   94, 2314,  534, 1092])


###


---



### Definição de uma Head

In [63]:
class Head (nn.Module): #Head = Attention
  def __init__(self):
    super().__init__()
    self.Query = nn.Linear (Embedding_Dimension, Head_Size, bias = False)
    self.Key = nn.Linear (Embedding_Dimension, Head_Size, bias = False)
    self.Value = nn.Linear (Embedding_Dimension, Head_Size, bias = False)

  def forward (self, x):
    B, T, C = x.shape

    Q = self.Query (x)
    K = self.Key (x)
    V = self.Value (x)

    attention_score = Q @ K.transpose (-2, -1) * (Head_Size ** -0.5)

    tril = torch.tril (torch.ones (T, T, device = x.device))
    attention_score = attention_score.masked_fill (tril == 0, float ("-inf"))

    attention_score = F.softmax (attention_score, dim = -1)

    output = attention_score @ V
    return output


###


---


### Definição de Multi Head

In [ ]:
class Multi_Head (nn.Module):
  def __init__(self):
    super().__init__()
    self.Concat = nn.ModuleList ([Head() for i in range (Number_Head)]) #Head Concatenation #Definição de Multi Head Attention
    self.Output_Projection = nn.Linear (Embedding_Dimension, Embedding_Dimension) #Necessário para misturar a informação das Heads #Desta forma volta também à dimensão (B, T, C)

  def forward (self, x):

    output = self.Output_Projection (torch.cat ([h (x) for h in self.Concat], dim = -1))
    return output


###


---


### Definição de um Block (Pre-LayerNorm Transformer)

Em modelos GPT, um Block é:

- Layer Norm
- Attention
- Residual Connection (Add)
- Layer Norm
- Feed Forward Neural Network
- Residual Connection (Add)

In [64]:
class Block (nn.Module):
  def __init__(self):
    super().__init__()

    self.LayerNorm_Attention = nn.LayerNorm (Embedding_Dimension)
    self.Masked_Multi_Head_Attention = Multi_Head()


    self.FeedForward_NeuralNet = nn.Sequential (
        nn.Linear (Embedding_Dimension, 4 * Embedding_Dimension),
        nn.ReLU(),
        nn.Linear (Embedding_Dimension * 4, Embedding_Dimension)
    )
    self.LayerNorm_FNN = nn.LayerNorm (Embedding_Dimension)

  def forward (self, x):
    Norm_Attention = self.Masked_Multi_Head_Attention (self.LayerNorm_Attention (x)) #Norm #LayerNorm
    Add_Attention = x + Norm_Attention #Add #Residual Connection
    x = Add_Attention

    Norm_FNN = self.FeedForward_NeuralNet (self.LayerNorm_FNN (x)) #Norm #LayerNorm
    Add_FNN = x + Norm_FNN #Add #Residual Connection
    x = Add_FNN

    return x


###


---


### Modelo Adamastor-GPT

In [65]:
class ADAMASTOR_GPT (nn.Module):
  def __init__(self):
    super().__init__()
    self.Embedding = nn.Embedding (Tokenizer.get_vocab_size(), Embedding_Dimension)
    self.Positional_Encoding = nn.Embedding (Sequence_Length, Embedding_Dimension)

    self.Blocks = nn.Sequential (
        Block(),
        Block(),
        Block(),
        Block()
    )

    self.Language_Modeling = nn.Linear (Embedding_Dimension, Tokenizer.get_vocab_size()) #Converte o vetor final do modelo em logits sobre o vocabulário inteiro.

  def forward (self, x):
    B, T = x.shape

    x = self.Embedding (x) + self.Positional_Encoding (torch.arange (T, device = x.device))

    x = self.Blocks(x)

    logits = self.Language_Modeling (x)

    return logits #Return de um tensor (4, 8, 5000)



###


---


### Treino do Modelo (Super-Importante)


In [ ]:
if __name__ == "__main__":

  modelo = ADAMASTOR_GPT().to(device) #Chamamos o modelo par a GPU

  optimizer = torch.optim.AdamW (modelo.parameters(), lr = 3e-4) #Usa-se AdamW ou SGD #Responsável por atualizar os pesos do modelo
  loss_function = nn.CrossEntropyLoss() #Função de perda que mede quão diferente está a previsão do modelo da resposta correta.

  for i in range (5000):

    x, y = Batch()
    x, y = x.to(device), y.to(device)

    logits = modelo (x)

    #print (logits.shape) #(16,32,5000)
    B, T, V = logits.shape #(Batch, Sequence Length, Vocab_Size)

    loss =  loss_function(
        logits.view (B * T, V),
        y.view (B * T)
    ) #Entrada que o Cross Entropy espera

    optimizer.zero_grad() #Zera os gradientes acumulados nos parâmetros.
    loss.backward() #Backpropagation

    optimizer.step() #Optimizer aplicado

    if i % 100 == 0: #Printa de 100 em 100
      print(f"Treino {i} = {loss.item()}")



8.877662658691406
8.903328895568848
8.876006126403809
8.870694160461426
8.835282325744629
8.765649795532227
8.810341835021973
8.78768539428711
8.865728378295898
8.891809463500977
8.851958274841309
8.828129768371582
8.819046020507812
8.854317665100098
8.7686128616333
8.84736442565918
8.840629577636719
8.806550025939941
8.793252944946289
8.797718048095703
8.75506591796875
8.752398490905762
8.739474296569824
8.803078651428223
8.775636672973633
8.683602333068848
8.729453086853027
8.736300468444824
8.64584732055664
8.798457145690918
8.702441215515137
8.688758850097656
8.676541328430176
8.712503433227539
8.72335433959961
8.686867713928223
8.673194885253906
8.64030647277832
8.604987144470215
8.630980491638184
8.650918006896973
8.684118270874023
8.675980567932129
8.621975898742676
8.571191787719727
8.617780685424805
8.592873573303223
8.513116836547852
8.563426971435547
8.611974716186523
8.565123558044434
8.623019218444824
8.564112663269043
8.558127403259277
8.576393127441406
8.440857887268066


In [68]:
def gerar_texto (modelo, indice, max_tokens):

    device = next(modelo.parameters()).device  #Vai buscar o modelo à GPU
    idx = idx.to(device)

    for _ in range(max_tokens):

        idx_cond = idx[:, -Sequence_Length:]

        logits = modelo(idx_cond)
        logits = logits[:, -1, :]

        probabilidade = F.softmax(logits, dim = -1)

        next_token = torch.multinomial(probabilidade, num_samples = 1)

        indice = torch.cat([idx, next_token], dim = 1)

    return indice

inicio = torch.zeros((1, 1), dtype = torch.long).to(device)
output = gerar_texto (modelo, inicio, 100)
print (output)


#Problema no Tokenizer
#Continuar a testar, escrever explicação do códigop


 nós  espalh -te  o
 íser escap ânim ênci E se  a ou  ,
 do  eito,  diz  o Castelh des,  Os olhos  largo  áb ont às vezes  o foi  ficou  ro  ante.  At « inda não  a lev na água  o Mouro  em ti  velh e. « perd Despois que  mina  as de  Porém  ra  gar cheir em vão  ocup vossos  etal  ador enganos  to,  os d longe  diz


In [ ]:
text = Tokenizer.decode(output[0].tolist())
print(text)

In [66]:

modelo = ADAMASTOR_GPT().to(device)


#print (modelo.get_parameter)
#sum(p.numel() for p in modelo.parameters())

#for name, param in modelo.named_parameters():
#    print(name, param.shape)
